# Tekne Dedektörü — Sadece Gri/Siyah-Beyaz (Mono) Görseller — Google Colab GPU Eğitimi

Bu notebook aynı tarifle (YOLOv8s, backbone donuk `freeze=10`, imgsz=640, patience=25)
ama datasetin **sadece gri/mono videolardan** oluşan alt kümesiyle eğitir (renkli klipler tamamen
çıkarıldı) — renkli-only denemenin tam tersi karşılaştırma noktası. Video bazlı split: aynı
videonun kareleri hem train hem val'de birden bulunmuyor.

Train: 6045, Val: 1080 (toplam 7125 görüntü, 23 video).

**Önce yapman gerekenler:**
1. Üstteki menüden **Çalışma zamanı (Runtime) > Çalışma zamanı türünü değiştir > T4 GPU** (veya A100/L4) seç.
2. `boat_mono_dataset_bundle.zip` dosyasını (Mac'indeki `depth-anything` klasöründe, ~396MB) Google Drive'ına yükle.
3. Aşağıdaki hücreleri sırayla çalıştır.

In [ ]:
# 1) GPU kontrolü
!nvidia-smi

In [ ]:
# 2) Google Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3) Zip dosyasını Drive'dan al ve aç
# Zip'i Drive'da farklı bir yere yüklediysen ZIP_PATH'i güncelle.
ZIP_PATH = '/content/drive/MyDrive/boat_mono_dataset_bundle.zip'

!rm -rf /content/work
!mkdir -p /content/work
!unzip -q "$ZIP_PATH" -d /content/work
!ls /content/work

In [ ]:
# 4) ultralytics kur
!pip install -q ultralytics

In [ ]:
# 5) dataset.yaml içindeki path'i Colab'daki yeni konuma göre düzelt
# (Mac'te: /Users/armin/Desktop/depth-anything/yolo_dataset_v4 idi)
import pathlib

yaml_path = pathlib.Path('/content/work/yolo_dataset_v4_mono/dataset.yaml')
content = yaml_path.read_text()
print('--- eski ---')
print(content)

new_content = content.replace(
    '/Users/armin/Desktop/depth-anything/yolo_dataset_v4_mono',
    '/content/work/yolo_dataset_v4'
)
yaml_path.write_text(new_content)
print('--- yeni ---')
print(yaml_path.read_text())

In [ ]:
# 6) backbone'un gercekten ilk 10 katmanda bittigini teyit et (Mac'te de dogrulanmisti)
from ultralytics import YOLO

_check = YOLO('/content/work/yolov8s.pt')
for i, layer in enumerate(_check.model.model):
    print(i, layer.__class__.__name__)

In [ ]:
# 7) Eğitim — freeze YOK (tam fine-tune), imgsz=960, batch=32, patience=25.
# tek fark: dataset artik SADECE gri/mono videolardan olusuyor (renkli kayitlar cikarildi) - renkli-only
# denemenin ayna karsiligi. Fresh baslatiyoruz (resume degil) ki ikisi ayni recipe uzerinden temiz karsilastirilsin.
from ultralytics import YOLO

model = YOLO('/content/work/yolov8s.pt')
results = model.train(
    data='/content/work/yolo_dataset_v4_mono/dataset.yaml',
    epochs=80,
    imgsz=960,
    device=0,
    batch=32,
    patience=25,
    project='/content/work/runs_boat_yolo',
    name='boat_v4s_frozen_mono',
    verbose=True,
)

In [ ]:
# 8) Eğitim koptuysa devam ettirmek için (7. hücre yerine bunu çalıştır):
# from ultralytics import YOLO
# model = YOLO('/content/work/runs_boat_yolo/boat_v4s_frozen_mono/weights/last.pt')
# results = model.train(resume=True)

In [ ]:
# 9) Bitince: sonuçları Drive'a kopyala
!mkdir -p /content/drive/MyDrive/boat_v4s_frozen_mono_results
!cp -r /content/work/runs_boat_yolo/boat_v4s_frozen_mono /content/drive/MyDrive/boat_v4s_frozen_mono_results/
print('Kopyalandı: Google Drive > boat_v4s_frozen_mono_results > boat_v4s_frozen_mono')

## Eğitim bitince Mac'ine geri alma

1. Drive'daki `boat_v4s_frozen_mono_results/boat_v4s_frozen_mono` klasörünü indir (`weights/best.pt` şart, geri kalanı - grafikler/results.csv - isteğe bağlı ama karşılaştırma için faydalı).
2. Bana zip'i ilet, ben `runs/detect/runs_boat_yolo/boat_v4s_frozen_mono/` altına yerleştirip renkli-only ve karışık denemelerle karşılaştırırım.

**Not:** Bu dataset (7125 görüntü) renkli-only'den (3575) büyük ama karışık tam datasetten (10700) küçük - ayrıca negatif oranı (%48) renkli-only'den (%26) çok daha yüksek, bu da precision/recall karşılaştırmasını etkileyebilir.